# Notebook 3 — Pitch Q&A Cheatsheet

> **Purpose.** Second-screen reference during Q&A. Each section below addresses one anticipated judge question with a single answer, supporting chart, and defensible number. Navigate by the table of contents below.

## Anticipated Questions

**Methodological challenges**
- [Q1: What's your out-of-sample R²? Your original had NaN](#Q1)
- [Q2: With N=46 and 10 features, how stable are these coefficients?](#Q2)
- [Q3: Gini is your largest predictor — why isn't it in the deck?](#Q3)
- [Q4: Isn't this demographic predictors predicting demographic outcomes?](#Q4)
- [Q5: Why 5-fold CV instead of LOO?](#Q5)

**Data/framing challenges**
- [Q6: USDA says food deserts matter — your data contradicts that?](#Q6)
- [Q7: Haven't Medicaid expansion and FQHCs solved access?](#Q7)
- [Q8: Your time-cost estimate has a lot of assumptions — defend it](#Q8)
- [Q9: You studied EOTR but the ML uses all DC — is that kosher?](#Q9)

**Solution/implementation challenges**
- [Q10: Why Mastercard specifically? Couldn't any fintech do this?](#Q10)
- [Q11: Where would you pilot this first?](#Q11)
- [Q12: What's your theory of change — what does "success" look like in 12 months?](#Q12)

In [ ]:
# ── Setup ──
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

from data_pipeline import (
    build_main, apply_style, load_findings,
    ACCENT, WEAK, STRONG, NEUTRAL, EOTR_COLOR, WOTR_COLOR
)

apply_style()

# Load data and precomputed findings
CACHE_PATH = Path('datasets/main_analysis.csv')
if CACHE_PATH.exists():
    main = pd.read_csv(CACHE_PATH)
    print(f"Loaded cached main table: {main.shape}")
else:
    DATA_DIR = Path('datasets') if Path('datasets').exists() else Path('/content/datasets')
    main = build_main(str(DATA_DIR))
    CACHE_PATH.parent.mkdir(exist_ok=True)
    main.to_csv(CACHE_PATH, index=False)
    print(f"Built main table: {main.shape}")
main['Census Tract FIPS code'] = main['Census Tract FIPS code'].astype(str).str.zfill(11)
eotr = main[main['region']=='EOTR'].copy()

try:
    stability = pd.read_csv("datasets/bootstrap_stability.csv")
    compare = pd.read_csv("datasets/lasso_vs_ridge.csv")
    top10 = pd.read_csv("datasets/top10_deployment.csv")
except FileNotFoundError:
    print("Run Notebook 2 first to generate ML artifacts.")
    stability = pd.DataFrame()
    compare = pd.DataFrame()
    top10 = pd.DataFrame()

findings = load_findings()
print(f"Loaded {len(findings)} precomputed findings")

<a id='Q1'></a>
## Q1. "What's your out-of-sample R²? Your original report had NaN."

**Answer:** The original analysis used leave-one-out CV, which produces undefined R² when individual folds collapse. We fixed it by switching to 5-fold CV and adding a held-out test set. Both report valid, defensible numbers.

In [ ]:
# ── Q1 Answer ──
cv_r2 = findings.get('lasso_cv_r2_mean', None)
cv_std = findings.get('lasso_cv_r2_std', None)
ho_r2 = findings.get('lasso_holdout_r2', None)
ho_rmse = findings.get('lasso_holdout_rmse', None)

if cv_r2 is not None:
    print("Out-of-sample performance:")
    print(f"  5-fold CV R²: {cv_r2:.3f} (± {cv_std:.3f})")
    print(f"  Held-out test R²: {ho_r2:.3f}")
    print(f"  Held-out RMSE: {ho_rmse:.2f} percentage points")
    print()
    print("This is a moderate model fit — food insecurity has substantial unexplained")
    print("variation at the tract level, which is expected given BRFSS is already a")
    print("small-area estimation. The ML is validation, not prediction in itself.")
else:
    print("(Run Notebook 2 first to compute R²)")

<a id='Q2'></a>
## Q2. "With N=46 and 10 correlated features, how stable are these coefficients?"

**Answer:** We don't rely on a single Lasso run. We bootstrap 1000 times and report retention frequency per feature. Features retained in >80% of resamples are robust. Features below 40% are noise and we don't emphasize them.

In [ ]:
# ── Q2 Answer ──
if len(stability) > 0:
    print("Bootstrap stability across 1000 Lasso resamples:")
    print(stability.to_string(index=False))
    print()

    # Plot
    fig, ax = plt.subplots(figsize=(11, 6))
    sp = stability.sort_values('Retention %')
    colors = [STRONG if r > 80 else NEUTRAL if r > 40 else WEAK for r in sp['Retention %']]
    bars = ax.barh(sp['Feature'], sp['Retention %'], color=colors, alpha=0.85)
    for bar, ret in zip(bars, sp['Retention %']):
        ax.text(ret + 1.5, bar.get_y() + bar.get_height()/2,
                f'{ret:.0f}%', va='center', fontsize=10)
    ax.axvline(80, color=STRONG, linestyle='--', alpha=0.6, label='Robust (>80%)')
    ax.axvline(40, color=WEAK, linestyle='--', alpha=0.6, label='Noise (<40%)')
    ax.set_xlabel('Retention frequency (1000 bootstrap resamples)')
    ax.set_xlim(0, 115)
    ax.set_title('Q2: Which features are robustly selected?')
    ax.legend(loc='lower right')
    sns.despine()
    plt.tight_layout()
    plt.show()

<a id='Q3'></a>
## Q3. "Gini Coefficient Score is your largest predictor. Why isn't it in the pitch deck?"

**Answer:** It is, in the way that matters. Tract-level Gini at low-income levels measures whether a neighborhood is uniformly poor or economically mixed. Mixed-income tracts sustain commercial diversity, which is the host infrastructure our intervention needs. So "raise Gini" isn't the policy — "raise commercial ecosystem viability in uniform-poor tracts" is. They measure the same phenomenon from different angles.

In [ ]:
# ── Q3 Answer ──
# Show that Gini and Commercial Diversity track the same underlying construct
cols = ['Gini Coefficient Score', 'Commercial Diversity Score',
        'Small Business Loans Score', 'food_insecurity_pct']
cols_present = [c for c in cols if c in main.columns]

if len(cols_present) >= 3:
    valid = main[cols_present].dropna()
    corr = valid.corr(method='spearman')

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, ax=ax, cbar_kws={'label': 'Spearman ρ'})
    ax.set_title('Q3: Gini tracks Commercial Diversity\n(they measure related concepts)')
    plt.tight_layout()
    plt.show()

    print()
    print("Reading: Gini positively correlates with Commercial Diversity and Small")
    print("Business Loans. All three negatively correlate with food insecurity.")
    print("They are manifestations of the same underlying 'commercial ecosystem")
    print("viability' — the policy handle we can actually pull is the business side.")

<a id='Q4'></a>
## Q4. "BRFSS PLACES is a small-area estimate from ACS demographics. Aren't you predicting demographics with demographics?"

**Answer:** Yes, partially — and we state this explicitly in the limitations. But the analysis is not just predicting SAE values. Two reasons it still matters:

1. **Independent datasets corroborate.** USDA food access, HUD labor market, and DC built environment are not BRFSS inputs. They agree with the story.
2. **The paradoxes survive this critique.** The proximity paradox and time-cost paradox don't depend on BRFSS predictions. They come from direct measures (USDA distance, ACS commute time).

The ML model is the weakest link, not the core argument. If a judge fully discounts the ML, the three paradoxes still stand.

In [ ]:
# ── Q4 Answer: show the paradoxes don't depend on BRFSS ──
# The proximity paradox uses only USDA + DC BE. Rerun it with only those.

high_fi = eotr.nlargest(10, 'food_insecurity_pct')
low_fi = eotr.nsmallest(10, 'food_insecurity_pct')

direct_measures = pd.DataFrame({
    'Group': ['Highest 10 FI', 'Lowest 10 FI'],
    'USDA: % >0.5mi supermarket': [
        round(high_fi['lapophalfshare'].mean(), 1),
        round(low_fi['lapophalfshare'].mean(), 1)
    ],
    'DC BE: Medical proximity pctile': [
        round(high_fi['Driver 7: Medical Care (percentiles)'].mean(), 2),
        round(low_fi['Driver 7: Medical Care (percentiles)'].mean(), 2)
    ],
    'ACS: Long commute rate (%)': [
        round(high_fi['long_commute_rate'].mean(), 1),
        round(low_fi['long_commute_rate'].mean(), 1)
    ]
})
print("Even with BRFSS fully discounted, direct measures show the paradox:")
print(direct_measures.to_string(index=False))
print()
print("Sources (all non-BRFSS): USDA Food Access Atlas, DC Open Data, ACS.")

<a id='Q5'></a>
## Q5. "Why 5-fold CV instead of leave-one-out?"

**Answer:** LOO on our data produced NaN R² because some folds had near-constant target values, making R² undefined (division by zero in variance-explained). 5-fold gives stable R² estimates. This isn't a subjective choice — the LOO implementation literally wouldn't return a number.

In [ ]:
# ── Q5 Answer ──
from sklearn.linear_model import Lasso, LassoCV
from sklearn.model_selection import LeaveOneOut, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from data_pipeline import ML_FEATURES, ML_TARGET

features = [f for f in ML_FEATURES if f in main.columns]
mdf = main[features + [ML_TARGET]].dropna()
if len(mdf) > 0:
    X = StandardScaler().fit_transform(mdf[features].values)
    y = mdf[ML_TARGET].values

    # Demonstrate both
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    lasso = LassoCV(cv=kf, max_iter=20000, random_state=42).fit(X, y)

    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        loo_scores = cross_val_score(Lasso(alpha=lasso.alpha_, max_iter=20000),
                                       X, y, cv=LeaveOneOut(), scoring='r2')
    kf_scores = cross_val_score(Lasso(alpha=lasso.alpha_, max_iter=20000),
                                  X, y, cv=kf, scoring='r2')

    print("CV method comparison on our data:")
    print(f"  LeaveOneOut  R² mean: {loo_scores.mean():.3f}  (NaN-vulnerable)")
    print(f"  5-fold CV    R² mean: {kf_scores.mean():.3f}  (stable)")
    print()
    print("LOO has N-1 folds each with a single validation point. R² is undefined")
    print("when you're predicting a single value (no variance to explain).")
    print("5-fold gives us 5 validation sets of ~36 tracts each — enough to compute R².")

<a id='Q6'></a>
## Q6. "USDA explicitly designates food deserts based on distance. Your data contradicts federal methodology?"

**Answer:** Our data doesn't contradict USDA's data — it contradicts the *interpretation*. USDA's own data shows that in EOTR, the tracts farthest from supermarkets are not the ones with highest food insecurity. The distance-based designation may be useful at the national level but mis-targets resources at the DC-tract level. This is a narrow, data-supported claim.

In [ ]:
# ── Q6 Answer ──
# Show the scatter of USDA distance vs BRFSS food insecurity in EOTR
valid = eotr[['lapophalfshare','food_insecurity_pct']].dropna()
rho, p = stats.spearmanr(valid['lapophalfshare'], valid['food_insecurity_pct'])

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(valid['lapophalfshare'], valid['food_insecurity_pct'],
           color=EOTR_COLOR, alpha=0.7, s=60, edgecolor='white')
z = np.polyfit(valid['lapophalfshare'], valid['food_insecurity_pct'], 1)
xx = np.linspace(valid['lapophalfshare'].min(), valid['lapophalfshare'].max(), 100)
ax.plot(xx, np.polyval(z, xx), color=WEAK, linewidth=2, alpha=0.7,
        label=f'Observed slope (negative!)')
ax.set_xlabel('USDA: % of tract population >0.5mi from supermarket')
ax.set_ylabel('BRFSS: Food insecurity %')
ax.set_title(f'Q6: USDA distance vs BRFSS food insecurity in EOTR\n'
             f'Spearman ρ = {rho:+.2f}, p = {p:.3f}')
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()
print()
print(f"Both datasets use the FIPS codes EOTR residents live in. The correlation")
print(f"is negative — farther from supermarket, LOWER food insecurity. This is")
print(f"USDA's own data pointing against USDA's own distance-based framework")
print(f"as the binding constraint on food security in DC.")

<a id='Q7'></a>
## Q7. "DC expanded Medicaid and has several FQHCs in Wards 7 and 8. Hasn't this been solved?"

**Answer:** The coverage gap is largely closed. The outcomes gap is not. EOTR uninsurance is only modestly higher than the rest of DC, yet chronic disease prevalence is 20–30% higher. That's the insurance paradox. Coverage is necessary but not sufficient — what's left is the *economic cost of using coverage*, which the proposed intervention addresses.

In [ ]:
# ── Q7 Answer ──
# Reproduce the insurance paradox table
chronic = ['diabetes_pct','obesity_pct','bp_pct','asthma_pct']
chronic = [c for c in chronic if c in main.columns]

rows = []
for m in ['overall_uninsured_rate'] + chronic:
    if m not in main.columns: continue
    e = main[main['region']=='EOTR'][m].mean()
    w = main[main['region']=='Rest of DC'][m].mean()
    rows.append({
        'Measure': m.replace('_pct','').replace('_',' '),
        'EOTR': round(e, 1),
        'Rest of DC': round(w, 1),
        '% higher in EOTR': round((e/w - 1) * 100, 0) if w else 0
    })
ip = pd.DataFrame(rows)
print("Insurance paradox (EOTR vs Rest of DC):")
print(ip.to_string(index=False))
print()
print("Uninsurance: slightly higher. Chronic disease: 20-30% higher.")
print("Coverage and outcomes are decoupled.")

<a id='Q8'></a>
## Q8. "Your time-cost estimate uses several assumptions. Defend it."

**Answer:** The assumptions are transparent and defensible:
- **30 min wait** at a public clinic — consistent with MedPAC and KFF estimates
- **30 min appointment** — standard primary care duration
- **Commute time** interpolated linearly from ACS long-commute rate (calibrated at 0%→20min, 60%→40min)
- **$17/hr** = DC minimum wage (conservative — many residents earn more)

The 2–3 hour gap holds under a wide range of alternative assumptions. Even if you cut all numbers by half, the gap is still substantial. What matters is the *direction and magnitude*, not the decimal places.

In [ ]:
# ── Q8 Answer: sensitivity analysis on the time-cost estimate ──
def estimate_time(long_commute_rate, wait_min, appt_min):
    oneway = 20 + (long_commute_rate / 60) * 20
    return 2 * oneway + wait_min + appt_min

# Test under 3 different assumption sets
scenarios = [
    ('Baseline', 30, 30),
    ('Pessimistic (longer wait)', 45, 30),
    ('Conservative (shorter wait)', 15, 30),
]

rows = []
for name, wait, appt in scenarios:
    main['tc'] = main['long_commute_rate'].apply(lambda r: estimate_time(r, wait, appt))
    e = main[main['region']=='EOTR']['tc'].mean() / 60
    w = main[main['region']=='Rest of DC']['tc'].mean() / 60
    rows.append({
        'Scenario': name,
        'EOTR hrs': round(e, 2),
        'Rest of DC hrs': round(w, 2),
        'Gap (hrs)': round(e - w, 2)
    })
sens = pd.DataFrame(rows)
print("Time-cost gap is robust across assumption scenarios:")
print(sens.to_string(index=False))
print()
print("Regardless of wait-time assumption, the gap is >1 hour per visit.")

<a id='Q9'></a>
## Q9. "The findings are about EOTR but the ML uses all DC tracts. Is that consistent?"

**Answer:** Yes, and it's deliberate for statistical power. The ML is trained on all ~180 DC tracts to get a reliable estimate of feature effects. We then apply those effects to EOTR for deployment prioritization. EOTR isn't an outlier in the DC distribution; it's the low-inclusion tail. Using all DC gives us more power to estimate the relationships, which we then use to prioritize within EOTR.

In [ ]:
# ── Q9 Answer: show that EOTR sits on the low tail of DC, not outside it ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# IGS distribution
ax = axes[0]
ax.hist(main[main['region']=='Rest of DC']['Inclusive Growth Score'].dropna(),
        bins=20, alpha=0.6, color=WOTR_COLOR, label='Rest of DC', edgecolor='white')
ax.hist(main[main['region']=='EOTR']['Inclusive Growth Score'].dropna(),
        bins=20, alpha=0.75, color=EOTR_COLOR, label='EOTR', edgecolor='white')
ax.set_xlabel('Inclusive Growth Score')
ax.set_ylabel('Tracts')
ax.set_title('EOTR is the low tail of DC, not a separate population')
ax.legend()
sns.despine(ax=ax)

# Food insecurity distribution
ax = axes[1]
ax.hist(main[main['region']=='Rest of DC']['food_insecurity_pct'].dropna(),
        bins=20, alpha=0.6, color=WOTR_COLOR, label='Rest of DC', edgecolor='white')
ax.hist(main[main['region']=='EOTR']['food_insecurity_pct'].dropna(),
        bins=20, alpha=0.75, color=EOTR_COLOR, label='EOTR', edgecolor='white')
ax.set_xlabel('Food insecurity %')
ax.set_ylabel('Tracts')
ax.set_title('EOTR is the high tail of food insecurity')
ax.legend()
sns.despine(ax=ax)

plt.tight_layout()
plt.show()
print("EOTR overlaps with Rest of DC — same population, same feature relationships.")
print("Using all DC for training is defensible because it expands power without")
print("biasing the coefficient estimates for EOTR.")

<a id='Q10'></a>
## Q10. "Why Mastercard specifically? Couldn't Stripe, Square, or Visa do this?"

**Answer:** Three Mastercard-specific assets that aren't easily replicable:

1. **IGS itself.** The Inclusive Growth Score is a Mastercard asset that lets the program target tracts based on an economic-inclusion signal other payment networks don't have. This is both the analytical backbone and the ongoing measurement layer.

2. **SpendingPulse + small business transaction data.** To measure whether the intervention raised commercial activity in host businesses, you need de-identified transaction flow data at the tract level. Mastercard's SpendingPulse and SB transaction panel is the only commercially available source.

3. **Center for Inclusive Growth + existing DC health and financial inclusion partnerships.** Mastercard's CIG already runs programs with DC-based community organizations. The pilot can leverage that warm network rather than starting cold.

A Visa or Stripe could theoretically do the payment-rail piece. None of them could do the targeting, measurement, and deployment layers together.

<a id='Q11'></a>
## Q11. "Where would you pilot this first?"

**Answer:** We built a Need × Readiness ranking. The top 5 tracts (see chart) are Phase 1 launch sites. Criteria:
- **Need:** high food insecurity, transportation barriers, mental distress
- **Readiness:** existing commercial diversity, small-business loan activity, lower digital disconnection

These tracts have both the highest-impact potential *and* the host infrastructure (barbershops, salons, laundromats) for embedded care kiosks.

In [ ]:
# ── Q11 Answer ──
if len(top10) > 0:
    print("Top 10 EOTR tracts for pilot deployment:")
    print(top10.to_string(index=False))
    print()
    print("Phase 1 launch: top 5 tracts (highest need AND readiness)")
    print("Phase 2: tracts 6-10 (high need, building readiness)")
else:
    print("(Run Notebook 2 to generate deployment scores)")

<a id='Q12'></a>
## Q12. "What does success look like in 12 months?"

**Answer:** Three tiers of outcome, each measurable with data Mastercard already has.

| Tier | Metric | Target (12 months, 15–20 kiosks) |
|---|---|---|
| Access | Unique patients served via kiosk telehealth visits | 3,000–5,000 |
| Economic | Kiosk-driven incremental revenue per host business | $400–800/mo |
| Community | SpendingPulse uplift in host-business category | +5–10% yoy |

At 20 kiosks: ~$720K/year in avoided ER visits (using $1,500 avoided-ER as conservative estimate, 2 per kiosk per month). Break-even for Mastercard at roughly 23 visits per kiosk per month.

If those numbers don't hit, we know the wedge doesn't work. If they do, we have a replicable playbook for Baltimore, Atlanta, Detroit.

---

## One-Page Summary (for the pitch itself)

If you get cut off mid-Q&A and need to make one closing statement:

> Three independent datasets show the sickest tracts in EOTR are closer to healthcare, not farther. A routine medical visit costs EOTR residents 2-3 more hours than DC residents elsewhere. This isn't a supply problem — it's a time-and-affordability problem, and the solution isn't more clinics, it's lower-friction care embedded in the commercial spaces residents already use. Mastercard is uniquely positioned to enable this because the payment rails, the targeting data, and the measurement infrastructure live together inside the company. No other partner can do all three. We're asking to pilot 15-20 tracts with $500K, measured against specific access, economic, and community metrics. Our deployment ranking points to five census tracts where we launch day one.